# BLS Processed

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Loading BLS Cleaned File

In [17]:
file_path = Path("../../../data/processed/BLS/BLS_2025_cleaned.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())

df.head()

Dataset shape: (444, 31)
Missing values: 0
Duplicates: 0


,area,industry,occupation,employment,employment_percent_relative_standard_error,annual_mean_wage,wage_percent_relative_standard_error,annual_10th_percentile_wage,annual_25th_percentile_wage,annual_median_wage,annual_75th_percentile_wage,annual_90th_percentile_wage,log_employment,annual_mean_wage_winsorized,annual_10th_percentile_wage_winsorized,annual_25th_percentile_wage_winsorized,annual_median_wage_winsorized,annual_75th_percentile_wage_winsorized,annual_90th_percentile_wage_winsorized,wage_spread,wage_progression_ratio,year,log_employment_normalized,annual_mean_wage_winsorized_normalized,annual_10th_percentile_wage_winsorized_normalized,annual_25th_percentile_wage_winsorized_normalized,annual_median_wage_winsorized_normalized,annual_75th_percentile_wage_winsorized_normalized,annual_90th_percentile_wage_winsorized_normalized,wage_spread_normalized,wage_progression_ratio_normalized
0,National,"Cross-Industry, Private, Federal, State, and L...",All Occupations (00-0000),155495730.0,0.0,69770.0,0.1,31200.0,37590.0,50980.0,80520.0,128560.0,18.862129,69770.0,31200.0,37590.0,50980.0,80520.0,128560.0,97360.0,4.120513,2025,3.418252,-0.098478,-0.715940,-0.662300,-0.449972,-0.107018,0.197986,0.335858,0.954452
1,National,"Cross-industry, Private Ownership only (000001)",All Occupations (00-0000),132782040.0,0.0,68930.0,0.1,30670.0,37030.0,48990.0,79110.0,129020.0,18.704220,68930.0,30670.0,37030.0,48990.0,79110.0,129020.0,98350.0,4.206717,2025,3.330502,-0.134758,-0.785912,-0.711566,-0.549986,-0.152217,0.208182,0.359561,1.046911
2,National,"Sector 11 - Agriculture, Forestry, Fishing and...",All Occupations (00-0000),413350.0,0.0,44590.0,0.4,33880.0,34630.0,36740.0,46420.0,63350.0,12.932052,44590.0,33880.0,34630.0,36740.0,46420.0,63350.0,29470.0,1.869835,2025,0.122886,-1.186019,-0.362119,-0.922709,-1.165652,-1.200141,-1.247381,-1.289620,-1.459540
3,National,Forestry and Logging (113000),All Occupations (00-0000),41620.0,0.1,56690.0,0.8,34670.0,44290.0,52640.0,63850.0,79040.0,10.636360,56690.0,34670.0,44290.0,52640.0,63850.0,79040.0,44370.0,2.279781,2025,-1.152839,-0.663412,-0.257821,-0.072860,-0.366543,-0.641398,-0.899615,-0.932872,-1.019848
4,National,Logging (113300),All Occupations (00-0000),41620.0,0.1,56690.0,0.8,34670.0,44290.0,52640.0,63850.0,79040.0,10.636360,56690.0,34670.0,44290.0,52640.0,63850.0,79040.0,44370.0,2.279781,2025,-1.152839,-0.663412,-0.257821,-0.072860,-0.366543,-0.641398,-0.899615,-0.932872,-1.019848


## Checking unique values for categorical features

In [18]:
categorical_columns = ["area", "industry", "occupation"]

for col in categorical_columns:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts().head(10))


area
Unique values: 1
area
National    444
Name: count, dtype: int64

industry
Unique values: 444
industry
Cross-Industry, Private, Federal, State, and Local (000000)           1
Cross-industry, Private Ownership only (000001)                       1
Sector 11 - Agriculture, Forestry, Fishing and Hunting (11--12)       1
Forestry and Logging (113000)                                         1
Logging (113300)                                                      1
Support Activities for Agriculture and Forestry (115000)              1
Support Activities for Crop Production (115100)                       1
Support Activities for Animal Production (115200)                     1
Sector 21 - Mining, Quarrying, and Oil and Gas Extraction (21--22)    1
Oil and Gas Extraction (211000)                                       1
Name: count, dtype: int64

occupation
Unique values: 1
occupation
All Occupations (00-0000)    444
Name: count, dtype: int64


## Extracting the SOC_CODES from occupation

In [19]:
df["soc_code"] = df["occupation"].str.extract(
    r"\((\d{2}-\d{4})\)"
)

print("Missing SOC codes:", df["soc_code"].isna().sum())

df[["occupation", "soc_code"]].head(10)

Missing SOC codes: 0


,occupation,soc_code
0,All Occupations (00-0000),00-0000
1,All Occupations (00-0000),00-0000
2,All Occupations (00-0000),00-0000
3,All Occupations (00-0000),00-0000
4,All Occupations (00-0000),00-0000
5,All Occupations (00-0000),00-0000
6,All Occupations (00-0000),00-0000
7,All Occupations (00-0000),00-0000
8,All Occupations (00-0000),00-0000
9,All Occupations (00-0000),00-0000


## Checking how many unique categories are in each categorical column

In [20]:
cardinality = pd.DataFrame({
    "feature": categorical_columns,
    "unique_categories": [
        df[col].nunique() for col in categorical_columns
    ]
})

cardinality

,feature,unique_categories
0,area,1
1,industry,444
2,occupation,1


## Extracting the industry codes

In [21]:
df["industry_code"] = df["industry"].str.extract(
    r"\(([^()]*)\)\s*$"
)

print("Missing industry codes:", df["industry_code"].isna().sum())

df[["industry", "industry_code"]].head(15)

Missing industry codes: 0


,industry,industry_code
0,"Cross-Industry, Private, Federal, State, and L...",000000
1,"Cross-industry, Private Ownership only (000001)",000001
2,"Sector 11 - Agriculture, Forestry, Fishing and...",11--12
3,Forestry and Logging (113000),113000
4,Logging (113300),113300
5,Support Activities for Agriculture and Forestr...,115000
6,Support Activities for Crop Production (115100),115100
7,Support Activities for Animal Production (115200),115200
8,"Sector 21 - Mining, Quarrying, and Oil and Gas...",21--22
9,Oil and Gas Extraction (211000),211000


## Extracting broader industry codes

In [22]:
df["industry_sector_code"] = df["industry_code"].str.extract(
    r"^(\d{2})"
)

df.loc[
    df["industry_code"].isin(["000000", "000001"]),
    "industry_sector_code"
] = pd.NA

print(df["industry_sector_code"].value_counts(dropna=False))

industry_sector_code
33     45
62     44
48     32
31     29
61     29
23     23
81     21
32     19
51     19
56     19
45     18
52     18
54     17
42     15
71     15
44     14
22     13
72     11
21      9
99      9
49      7
53      7
11      6
55      3
NaN     2
Name: count, dtype: int64


## One-hot encoding the industry codes

In [23]:
df_encoded = pd.get_dummies(
    df,
    columns=["industry_sector_code"],
    prefix="sector",
    dtype=int
)

print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)

df_encoded.filter(regex=r"^sector_").head()

Original shape: (444, 34)
Encoded shape: (444, 57)


,sector_11,sector_21,sector_22,sector_23,sector_31,sector_32,sector_33,sector_42,sector_44,sector_45,sector_48,sector_49,sector_51,sector_52,sector_53,sector_54,sector_55,sector_56,sector_61,sector_62,sector_71,sector_72,sector_81,sector_99
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Adding a cross-industry indicator 
What each category the code belongs to 

In [24]:
df_encoded["sector_cross_industry"] = (
    df_encoded["industry_code"].isin(["000000", "000001"])
).astype(int)

print("Cross-industry rows:", df_encoded["sector_cross_industry"].sum())
print("Final shape:", df_encoded.shape)

Cross-industry rows: 2
Final shape: (444, 58)


## Verifying the one-hot encoding worked

In [25]:
sector_columns = [
    col for col in df_encoded.columns
    if col.startswith("sector_")
]

print("Rows with exactly one sector:", 
      (df_encoded[sector_columns].sum(axis=1) == 1).sum())

print("Rows with missing sector assignments:",
      (df_encoded[sector_columns].sum(axis=1) == 0).sum())

Rows with exactly one sector: 444
Rows with missing sector assignments: 0


## Saving the final output

In [26]:
output_path = Path("../../data/processed/BLS/BLS_2025_processed.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_encoded.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", df_encoded.shape)

Saved: ..\..\data\processed\BLS\BLS_2025_processed.csv
Final shape: (444, 58)


## Verifying the final csv

In [27]:
df_check = pd.read_csv(output_path)

print("Shape:", df_check.shape)
print("Missing values:", df_check.isna().sum().sum())
print("Duplicates:", df_check.duplicated().sum())

Shape: (444, 58)
Missing values: 0
Duplicates: 0
